# PMAPS Workshop: IDAES-GTEP, Session 2

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/idaes/idaes-gtep.git/main?urlpath=%2Fdoc%2Ftree%2Fdocs%2Fsource%2Ftutorials%2F123bus%2Ftutorial_123bus.ipynb)

Welcome! In this tutorial, we will demonstrate using IDAES Generation and Transmission Expansion Planning (GTEP) with a 123-bus system in Texas.

As we step through the notebook, you should notice that all the steps to set up and solve a model are the same as in the simpler 5-bus case. The only major difference is the scale of the system reflected in the data files. However, we will demonstrate some helpful (optional) functionality for larger cases that take longer to solve.

In [ ]:
# We'll again need to install a solver. Run this cell, then restart your notebook and continue.
%pip install highspy

In [ ]:
# suppressing some logs/warnings
import logging
import warnings
logging.getLogger().setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

## Setting up and solving the model

First, we create our `ExpansionPlanningData` instance and use `load_prescient` to load the relevant data files.

For our baseline case, we will have 2 representative periods to capture variation in loads and renewable generation. We will keep the rest of the setup relatively simple: one investment period, one commitment period, one dispatch period, and default configuration options (except for `scale_loads=False`).

In [ ]:
from pathlib import Path
from gtep.gtep_data import ExpansionPlanningData

data_path = Path("../../../../gtep/data/123_Bus_Resil_Week")

rep_days=[
    "2019-01-28 00:00",
    "2019-06-10 00:00",
]
rep_weights={
    "2019-01-28 00:00": 140,
    "2019-06-10 00:00": 225,
} 

data_object = ExpansionPlanningData(
    stages=1,
    num_reps=2,
    num_commit=1,
    num_dispatch=1,
    duration_representative_period=1,
)
data_object.load_prescient(
    data_path,
    representative_dates=rep_days,
    representative_weights=rep_weights,
)

Next we read in cost data using `DataProcessing`. This step is exactly the same as the 5-bus case:

In [ ]:
from gtep.gtep_data_processing import DataProcessing

bus_data_path = Path(
    "../../../../gtep/data/costs/Bus_data_gen_weights_mappings.csv"
)
cost_data_path = Path(
    "../../../../gtep/data/costs/2022_v3_Annual_Technology_Baseline_Workbook_Mid-year_update_2-15-2023_Clean.xlsx"
)
ng_cost_path = Path(
    "../../../../gtep/data/costs/Total_Energy_Supply_Disposition_and_Price_Summary.csv"
)

candidate_gens = [
    "Natural Gas_FE",
    "Solar - Utility PV",
    "Land-Based Wind",
]

cost_data = DataProcessing()
cost_data.load_gen_data(
    bus_data_path=bus_data_path,
    cost_data_path=cost_data_path,
    ng_cost_path=ng_cost_path,
    candidate_gens=candidate_gens,
)

Finally, we create and solve the model. However, we'll do two small (optional) things differently since the 123-bus case takes longer to solve than the 5-bus case:

1. Since the 123-bus case is much larger than the 5-bus case, it can be useful to track how much time elapses between each step (build, transform, and solve). To do so, we can make use of the `ExpansionPlanningModel`'s `.timer` attribute, which is a `TicTocTimer` instance (see the Pyomo docs for it here: https://pyomo.readthedocs.io/en/stable/api/pyomo.common.timing.TicTocTimer.html). In short, calling `mod_object.timer.toc()` will report the elapsed time since the last `.tic()`/`.toc()`. Note that an `ExpansionPlanningModel` automatically calls `self.timer.tic()` when `.create_model()` is called on it (with the message `"Creating GTEP Model"`).
2. Passing configuration options to the solver is a good way to help the model solve faster. For the sake of the tutorial, we pass `opt.options["mip_rel_gap"] = 0.01` to HiGHS, which allows the solver to terminate once the relative gap between the best feasible solution found and the best bound on the optimal solution is below 1%. Other useful options include scaling (such as `user_objective_scale` and `user_bound_scale`).

In [ ]:
from gtep.gtep_model import ExpansionPlanningModel
from pyomo.environ import SolverFactory, TransformationFactory
from contextlib import redirect_stdout, redirect_stderr
import time  # [instrument]
import resource  # [instrument]
from pyomo.util.model_size import build_model_size_report  # [instrument]

mod_object = ExpansionPlanningModel(
    data=data_object,
    cost_data=cost_data,
    config={"scale_loads": False},
)
mod_object.create_model()
mod_object.timer.toc("Finished model build")

TransformationFactory("gdp.bigm").apply_to(mod_object.model)
mod_object.timer.toc("Finished model transformation")
size_baseline = build_model_size_report(mod_object.model)  # [instrument]

opt = SolverFactory("highs")
opt.options["mip_rel_gap"] = 0.01

_t0 = time.perf_counter()  # [instrument]
with open("output.log", "w") as f, redirect_stdout(f), redirect_stderr(f):
    result = opt.solve(mod_object.model, tee=True)
wall_baseline = time.perf_counter() - _t0  # [instrument]

mod_object.timer.toc("Finished solving");
print("[instrument] baseline termination:", result["Solver"][0]["Termination condition"])
print(f"[instrument] baseline solve wall (s): {wall_baseline:.2f}")
print(size_baseline)


Now we use the `ExpansionPlanningSolution` class to write out the results and generate plots:

In [ ]:
from copy import copy
from IPython.display import display, HTML
from gtep.gtep_solution import ExpansionPlanningSolution

def display_plotly_as_html(fig, title=None):
    if title is not None:
        fig = copy(fig)
        fig.update_layout(title=title)
    fig_html = fig.to_html(full_html=False, include_plotlyjs="cdn")
    return display(HTML(fig_html))

# create solution object and write out to json
soln = ExpansionPlanningSolution(data_path)
soln_path = Path("soln")
soln.save_results_in_json_files(mod_object, soln_path)

# perform plotting
pie = soln.create_plots("combined", soln_path, data_path, "piechart", savefig=False)
stackgraph = soln.create_stackgraph(soln_path, rep_days, savefig=False)

display_plotly_as_html(pie, title="Baseline 123-bus case")
display_plotly_as_html(stackgraph, title="Baseline 123-bus case")

From the pie chart, we can see that Gas CT makes up the majority of the installed generation capacity, and it also makes up a large portion of generation in the stackgraph. Wind candidates were the only ones to be installed and used to meet the load.

To free up some memory, we can `del` some objects we no longer need:

In [ ]:
import gc
from pyomo.environ import value

baseline_objective = value(mod_object.model.total_cost_objective)
print(f"[instrument] baseline_objective: {baseline_objective:,.4f}")  # [instrument]
metrics = {  # [instrument]
    "baseline": {
        "objective": baseline_objective,
        "termination": str(result["Solver"][0]["Termination condition"]),
        "wall": wall_baseline,
        "size": size_baseline,
        "log": "output.log",
        "soln": "soln",
    }
}
del mod_object, opt
gc.collect();


## Experiments

Here we can define a helper function to solve the model and return plots:

In [ ]:
from pyomo.environ import value

def solve_model_and_make_plots(
    model_object: ExpansionPlanningModel,
    data_path: Path|str,
    write_dir: Path|str,
    tee: bool=True,
    solver_options: None|dict=None,
    log_file: str = "output.log",  # [instrument] per-run HiGHS log
):

    # transform and solve
    TransformationFactory("gdp.bigm").apply_to(model_object.model)
    model_object.timer.toc("Finished model transformation")
    size = build_model_size_report(model_object.model)  # [instrument]

    opt = SolverFactory("highs")
    if solver_options is not None:
        for option, val in solver_options.items():
            opt.options[option] = val

    _t0 = time.perf_counter()  # [instrument]
    with open(log_file, "w") as f, redirect_stdout(f), redirect_stderr(f):
        result_object = opt.solve(model_object.model, tee=tee)
    wall = time.perf_counter() - _t0  # [instrument]
    model_object.timer.toc("Finished solving")

    # check termination condition
    term_cond = result_object["Solver"][0]["Termination condition"]
    print("Termination condition:", term_cond)
    print(f"[instrument] {write_dir} solve wall (s): {wall:.2f}")
    print(size)
    # [instrument] a time-limited run with a feasible incumbent is still
    # written out (the legacy HiGHS wrapper loads the incumbent); only a run
    # with nothing to report returns early.
    if term_cond != "optimal" and str(term_cond) != "maxTimeLimit":
        return None, None, result_object, wall, size

    # make solution
    soln_object = ExpansionPlanningSolution(data_path)
    soln_object_path = (Path() / write_dir).resolve()
    soln_object.save_results_in_json_files(model_object, soln_object_path)
    model_object.timer.toc("Finished saving results to json")
    pie = soln_object.create_plots(
        "combined", soln_object_path, data_path, "piechart", savefig=False
    )
    rep_days = [
        value(model_object.model.representativeDate[idx])
        for idx in model_object.model.representativeDate
    ]
    stackgraph = soln_object.create_stackgraph(
        soln_object_path, rep_days, savefig=False
    )
    model_object.timer.toc("Finished plotting")
    pie.write_html(f"{write_dir}_pie.html")  # [instrument]
    stackgraph.write_html(f"{write_dir}_stackgraph.html")  # [instrument]
    return pie, stackgraph, result_object, wall, size


### Experiment 1

In this experiment, we keep the number of time periods the same, but we set `include_investment=False`. This means that candidate assets cannot be invested in, and the model must do what it can with what is already installed.

To keep our memory usage within Binder's limits, we will just show the code below (but not actually build or solve the model). Then, we will load the HTML pie chart from file to show the resulting generation mix.

```python
mod_object_expr1 = ExpansionPlanningModel(
    data=data_object,  # same data object
    cost_data=cost_data,  # same cost data object
    config={"scale_loads": False, "include_investment": False},
)
mod_object_expr1.create_model()
mod_object_expr1.timer.toc("Finished model build")

# for this experiment, we'll just look at the pie chart
pie_exper1, _ = solve_model_and_make_plots(
    mod_object_expr1,
    data_path,
    "soln_expr1",
    solver_options={"mip_rel_gap": 0.01}
)

print(
    "Objective for baseline:",
    f"{baseline_objective :,.0f}",
)
expr1_objective = value(mod_object_expr1.model.total_cost_objective)
print(
    "Objective for experiment 1:",
    f"{expr1_objective :,.0f}",
)

display_plotly_as_html(pie, title="Baseline")
display_plotly_as_html(
    pie_exper1,
    title="Experiment 1 (no investment)",
)
```

In [ ]:
# [instrument] Experiment 1 run live (code from the markdown cell above),
# written to soln_expr1_local/ so the shipped soln_expr1/ stays the reference.
mod_object_expr1 = ExpansionPlanningModel(
    data=data_object,  # same data object
    cost_data=cost_data,  # same cost data object
    config={"scale_loads": False, "include_investment": False},
)
mod_object_expr1.create_model()
mod_object_expr1.timer.toc("Finished model build")

pie_exper1, _, result_expr1, wall_expr1, size_expr1 = solve_model_and_make_plots(
    mod_object_expr1,
    data_path,
    "soln_expr1_local",
    solver_options={"mip_rel_gap": 0.01},
    log_file="highs_expr1.log",
)

print(
    "Objective for baseline:",
    f"{baseline_objective :,.0f}",
)
expr1_objective = value(mod_object_expr1.model.total_cost_objective)
print(
    "Objective for experiment 1:",
    f"{expr1_objective :,.0f}",
)
print(f"[instrument] expr1_objective: {expr1_objective:,.4f}")
metrics["expr1"] = {
    "objective": expr1_objective,
    "termination": str(result_expr1["Solver"][0]["Termination condition"]),
    "wall": wall_expr1,
    "size": size_expr1,
    "log": "highs_expr1.log",
    "soln": "soln_expr1_local",
}

display_plotly_as_html(pie, title="Baseline")
display_plotly_as_html(
    pie_exper1,
    title="Experiment 1 (no investment, live)",
)

del mod_object_expr1
gc.collect();


In [ ]:
pie_exper1_html = HTML(filename="soln_expr1/piechart_combined_2019.html")

display_plotly_as_html(pie, title="Baseline")
display(pie_exper1_html)

As an alternative to the stackgraph, we can manually extract results from the `generation.json` file produced by `ExpansionPlanningSolution`:

In [ ]:
import json
import pandas as pd

# get a dictionary that maps generator ID to unit type
gen_unit_type_mapping = pd.read_csv(
    data_path / "gen.csv", index_col="GEN UID"
)["Unit Type"].to_dict()

def total_gen_by_type(path_to_gen_result: Path, dispatch_duration: float):

    with open(path_to_gen_result, "r") as f:
        generation_data = json.load(f)  # in MWh

    totals_by_unit_type = {}

    # loop through results, computing total energy by unit type
    for key, val in generation_data.items():

        # gen UID is at the very end of the variable name
        gen_uid = key.split(".")[-1]

        # apply mapping
        unit_type = gen_unit_type_mapping[gen_uid]

        # compute energy from power
        gen_gwh = val * dispatch_duration / 1000

        # add unit type to dict if isn't already there
        if unit_type not in totals_by_unit_type:
            totals_by_unit_type[unit_type] = 0

        totals_by_unit_type[unit_type] += gen_gwh

    # sort alphabetically by unit type and print
    print({
        unit_type: totals_by_unit_type[unit_type]
        for unit_type in sorted(totals_by_unit_type)
    })

print("-" * 40)
print("Baseline")
total_gen_by_type(Path("soln/generation.json"), dispatch_duration=1.0)  # dispatch duration is 1 hour

print("-" * 40)
print("Experiment 1")
total_gen_by_type(Path("soln_expr1/generation.json"), dispatch_duration=1.0)

A few observations from these results:
- From the pie charts, we can see that Wind makes up a smaller proportion of the generation mix in the experiment (since the model cannot invest in new Wind capacity).
- From the printed numerical results, we see that Coal and Gas CT generation increased in the experiment.

### Experiment 2

In this experiment, we showcase a more complex model setup with 10 representative periods and 5 commitment periods. We also pass the `transmission=True` configuration option.

Since model takes around 36 minutes to run (and again, to manage our memory usage), we will show how to set it up in markdown, but won't actually build the model or solve it. Instead, we will load HTML figures and display them.

Building the model:

```python
data_object_exper2 = ExpansionPlanningData(
    stages=1,
    num_reps=10,
    num_commit=5,
    num_dispatch=1,
    duration_representative_period=5,
)

data_object_exper2.load_prescient(
    data_path,
    representative_dates=[
        "2019-01-28 00:00",
        "2019-03-07 00:00",
        "2019-03-16 00:00",
        "2019-05-10 00:00",
        "2019-06-10 00:00",
        "2019-06-30 00:00",
        "2019-07-15 00:00",
        "2019-10-29 00:00",
        "2019-11-03 00:00",
        "2019-12-10 00:00",
    ],
    representative_weights={
        "2019-01-28 00:00": 45,
        "2019-03-07 00:00": 35,
        "2019-03-16 00:00": 25,
        "2019-05-10 00:00": 35,
        "2019-06-10 00:00": 30,
        "2019-06-30 00:00": 35,
        "2019-07-15 00:00": 45,
        "2019-10-29 00:00": 35,
        "2019-11-03 00:00": 40,
        "2019-12-10 00:00": 40,
    },
)

mod_object_expr2 = ExpansionPlanningModel(
    data=data_object_exper2,
    cost_data=cost_data,  # same cost data
    config={"scale_loads": False, "transmission": True},
)
mod_object_expr2.create_model()
mod_object_expr2.timer.toc("Finished model build")
```

In [ ]:
# [instrument] Experiment 2 built and solved live (code from the markdown
# cell above), written to soln_expr2_local/, mip_rel_gap 0.01, time_limit 7200 s.
data_object_exper2 = ExpansionPlanningData(
    stages=1,
    num_reps=10,
    num_commit=5,
    num_dispatch=1,
    duration_representative_period=5,
)

data_object_exper2.load_prescient(
    data_path,
    representative_dates=[
        "2019-01-28 00:00",
        "2019-03-07 00:00",
        "2019-03-16 00:00",
        "2019-05-10 00:00",
        "2019-06-10 00:00",
        "2019-06-30 00:00",
        "2019-07-15 00:00",
        "2019-10-29 00:00",
        "2019-11-03 00:00",
        "2019-12-10 00:00",
    ],
    representative_weights={
        "2019-01-28 00:00": 45,
        "2019-03-07 00:00": 35,
        "2019-03-16 00:00": 25,
        "2019-05-10 00:00": 35,
        "2019-06-10 00:00": 30,
        "2019-06-30 00:00": 35,
        "2019-07-15 00:00": 45,
        "2019-10-29 00:00": 35,
        "2019-11-03 00:00": 40,
        "2019-12-10 00:00": 40,
    },
)

_t_build0 = time.perf_counter()
mod_object_expr2 = ExpansionPlanningModel(
    data=data_object_exper2,
    cost_data=cost_data,  # same cost data
    config={"scale_loads": False, "transmission": True},
)
mod_object_expr2.create_model()
build_expr2 = time.perf_counter() - _t_build0
mod_object_expr2.timer.toc("Finished model build")

_t_tr0 = time.perf_counter()
pie_exper2, stackgraph_exper2, result_expr2, wall_expr2, size_expr2 = solve_model_and_make_plots(
    mod_object_expr2,
    data_path,
    "soln_expr2_local",
    solver_options={"mip_rel_gap": 0.01, "time_limit": 7200},
    log_file="highs_expr2.log",
)
transform_expr2 = time.perf_counter() - _t_tr0 - wall_expr2  # transform + post-solve I/O

expr2_objective = value(mod_object_expr2.model.total_cost_objective)
print(f"[instrument] expr2_objective: {expr2_objective:,.4f}")
print(f"[instrument] expr2 build (s): {build_expr2:.2f}")
peak_rss_gb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e9
print(f"[instrument] peak RSS (GB): {peak_rss_gb:.2f}")
metrics["expr2"] = {
    "objective": expr2_objective,
    "termination": str(result_expr2["Solver"][0]["Termination condition"]),
    "wall": wall_expr2,
    "size": size_expr2,
    "log": "highs_expr2.log",
    "soln": "soln_expr2_local",
    "build": build_expr2,
    "peak_rss_gb": peak_rss_gb,
}
if pie_exper2 is not None:
    display_plotly_as_html(pie, title="Baseline")
    display_plotly_as_html(pie_exper2, title="Experiment 2 (live)")
    display_plotly_as_html(stackgraph_exper2, title="Experiment 2 (live)")

del mod_object_expr2
gc.collect();


In [ ]:
pie_exper2_html = HTML(filename="soln_expr2/piechart_combined_2019.html")
stackgraph_exper2_html = HTML(filename="soln_expr2/stackgraph_generators.html")

display_plotly_as_html(pie, title="Baseline")
display(pie_exper2_html)
display_plotly_as_html(stackgraph, title="Baseline")
display(stackgraph_exper2_html)

In [ ]:
# [instrument] metrics tables -> metrics_123bus.md
import re, platform, sys, importlib.metadata as _md
import highspy, pyomo

gen_df = pd.read_csv(data_path / "gen.csv", index_col="GEN UID")
pmax = gen_df["PMax MW"].to_dict()


def gen_by_type_dict(path_to_gen_result, dispatch_duration=1.0):
    with open(path_to_gen_result) as f:
        generation_data = json.load(f)
    totals = {}
    for key, val in generation_data.items():
        ut = gen_unit_type_mapping[key.split(".")[-1]]
        totals[ut] = totals.get(ut, 0.0) + val * dispatch_duration / 1000
    return {k: totals[k] for k in sorted(totals)}


def parse_highs_log(path):
    txt = open(path).read()
    out = {}
    for label, pat in (
        ("gap", r"^\s+Gap\s+(.*)$"),
        ("nodes", r"^\s+Nodes\s+(\d+)"),
        ("timing", r"^\s+Timing\s+([\d.]+)"),
        ("status", r"^\s+Status\s+(.*)$"),
        ("lp_iters", r"^\s+LP iterations\s+(\d+)"),
    ):
        m = re.search(pat, txt, re.M)
        out[label] = m.group(1).strip() if m else "n/a"
    return out


def investments(soln_dir):
    """Non-zero investment-status entries from the solution JSONs."""
    d = {}
    for p in sorted(Path(soln_dir).glob("*.json")):
        if "invest" in p.name.lower():
            d.update(json.load(open(p)))
    return d


def candidate_summary(soln_dir):
    """-c candidates with a non-zero investment-status value, by variable."""
    rows = {}
    for key, val in investments(soln_dir).items():
        var, gen = key.rsplit(".", 1)
        var = var.split(".")[-1]
        if gen.endswith("-c"):
            rows.setdefault(var, []).append((gen, val))
    return rows


def new_capacity_by_type(soln_dir):
    """Installed candidate capacity (MW) by unit type."""
    cap = {}
    for key, val in investments(soln_dir).items():
        var, gen = key.rsplit(".", 1)
        var = var.split(".")[-1]
        if not gen.endswith("-c") or "Installed" not in var:
            continue
        ut = gen_unit_type_mapping[gen]
        mw = val if var.startswith("renewable") else pmax[gen] * val
        cap[ut] = cap.get(ut, 0.0) + mw
    return {k: cap[k] for k in sorted(cap)}


stored = {
    "baseline": {"COAL": 6.4307, "CT": 22.402249999999995, "WIND": 46.076880000000024},
    "expr1": {"COAL": 12.154599999999995, "CT": 39.881330000000034, "WIND": 22.873900000000003},
}
energy = {k: gen_by_type_dict(Path(m["soln"]) / "generation.json") for k, m in metrics.items()}
logs = {k: parse_highs_log(m["log"]) for k, m in metrics.items()}

lines = []
lines.append(
    f"Python {sys.version.split()[0]}, highspy {_md.version('highspy')}, "
    f"HiGHS {highspy.Highs().version()}, pyomo {pyomo.version.version}, "
    f"{platform.machine()} / {platform.platform()}"
)
lines += ["", "## T1 comparison", "",
          "| case | quantity | stored | obtained | rel. diff | termination | MIP gap | solve s |",
          "|---|---|---|---|---|---|---|---|"]
for case in ("baseline", "expr1"):
    m, e, lg = metrics[case], energy[case], logs[case]
    types = sorted(set(stored[case]) | set(e))
    for ut in types + ["total"]:
        s = sum(stored[case].values()) if ut == "total" else stored[case].get(ut, 0.0)
        o = sum(e.values()) if ut == "total" else e.get(ut, 0.0)
        rd = (o - s) / s if s else float("nan")
        lines.append(
            f"| {case} | {ut} GWh | {s:.4f} | {o:.4f} | {rd:+.2%} | "
            f"{m['termination']} | {lg['gap']} | {m['wall']:.2f} |"
        )
lines += ["", "## Objectives", ""]
for k, m in metrics.items():
    lines.append(f"- {k}: {m['objective']:,.4f} ({m['termination']}, gap {logs[k]['gap']})")

lines += ["", "## Model size / time", "",
          "| case | rows | cols | binaries | solve wall s | HiGHS timing s | nodes | LP iters | termination | gap |",
          "|---|---|---|---|---|---|---|---|---|---|"]
for k, m in metrics.items():
    a, lg = m["size"].activated, logs[k]
    lines.append(
        f"| {k} | {a.constraints} | {a.variables} | {a.binary_variables} | {m['wall']:.1f} | "
        f"{lg['timing']} | {lg['nodes']} | {lg['lp_iters']} | {m['termination']} | {lg['gap']} |"
    )
if "expr2" in metrics:
    lines.append(
        f"\nexpr2 build {metrics['expr2']['build']:.1f} s; peak RSS {metrics['expr2']['peak_rss_gb']:.2f} GB "
        f"(whole kernel, cumulative)."
    )

lines += ["", "## Energy by type (GWh, dispatch_duration=1.0)", ""]
for k, e in energy.items():
    lines.append(f"- {k}: {e}")

# per-generator diff, Experiment 1 vs shipped
shipped = json.load(open("soln_expr1/generation.json"))
local = json.load(open("soln_expr1_local/generation.json"))
keys = set(shipped) | set(local)
deltas = {k: local.get(k, 0.0) - shipped.get(k, 0.0) for k in keys}
big = sorted(deltas.items(), key=lambda kv: -abs(kv[1]))
lines += ["", "## Experiment 1 per-generator diff (local - shipped, MW)", "",
          f"- keys: shipped {len(shipped)}, local {len(local)}, union {len(keys)}",
          f"- max |delta|: {max(abs(v) for v in deltas.values()):.3f} MW",
          f"- generators with |delta| > 1 MW: {sum(abs(v) > 1 for v in deltas.values())}",
          "- five largest:"]
for k, v in big[:5]:
    lines.append(f"  - {k.split('.',1)[1] if '.' in k else k}: {v:+.3f} (shipped {shipped.get(k,0):.3f}, local {local.get(k,0):.3f})")

lines += ["", "## Candidate (-c) status in the baseline (non-zero entries)", ""]
for var, rows in candidate_summary("soln").items():
    lines.append(f"- {var}: {len(rows)} candidates, e.g. " + "; ".join(f"{g}={v:.1f}" for g, v in rows[:6]))

if "expr2" in metrics:
    lines += ["", "## Experiment 2 investment summary (installed candidate MW by type)", "",
              f"- {new_capacity_by_type('soln_expr2_local')}",
              "", "## Experiment 2 candidate status (non-zero entries)", ""]
    for var, rows in candidate_summary("soln_expr2_local").items():
        lines.append(f"- {var}: {len(rows)} candidates, e.g. " + "; ".join(f"{g}={v:.1f}" for g, v in rows[:6]))
lines += ["", "## Baseline investment summary (installed candidate MW by type)", "",
          f"- {new_capacity_by_type('soln')}"]

report = "\n".join(lines)
print(report)
with open("metrics_123bus.md", "w") as f:
    f.write(report + "\n")


A few observations:
- In the pie charts, generation capacities are similar to the baseline case. However, we invest in more Wind and Solar in the experiment.
- The finer time resolution is apparent in the experiment's stackgraph.